<a href="https://colab.research.google.com/github/GuillemD/TFM/blob/main/TFM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Predicción de ventas anuales de videojuegos en función del mes de lanzamiento
#### Guillem Dominguez, Sofia Gallardo, Carlos Pamplona

In [ ]:
# Librerías de tratamiento de datos
import numpy as np
import pandas as pd

# Librerías para visualización
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

# Librerías de Machine Learning
import sklearn

In [ ]:
#FUNCIONES


def parse_estimated_owners(owner_val):
    """Convierte la variable "estimated_owners" de string a tupla de enteros.

    Parámetros:
        owner_val (any): String con el rango de "estimated_owners".

    Retorna:
        tupla: Dos enteros que marcan el rango de "estimated_owners".
    """
    # 1. Si ya es una tupla con 2 elementos de tipo numérico, la conservamos tal cual
    if isinstance(owner_val, tuple) and len(owner_val) == 2:
        if isinstance(owner_val[0], (int, float)) and isinstance(owner_val[1], (int, float)):
            return (int(owner_val[0]), int(owner_val[1]))

    # 2. Si es nulo o no es una cadena de texto, devolvemos un rango por defecto (0, 0)
    if pd.isna(owner_val) or not isinstance(owner_val, str):
        return (0, 0)

    try:
        # 3. Limpiamos y dividimos la cadena por el guion '-'
        parts = owner_val.split('-')
        if len(parts) == 2:
            low = int(parts[0].strip().replace(',', ''))
            high = int(parts[1].strip().replace(',', ''))
            return (low, high)
    except Exception:
        pass

    return (0, 0)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Cargando el fichero JSON que contiene la información correcta en la cabecera

file_path = "/content/drive/MyDrive/TFM_DS_EquipoA/Colab/Datasets/games.json"

steam_data = pd.read_json(file_path)

In [ ]:
# Trasponemos el dataframe para que nos queden las columnas verticales

steam_dataT = steam_data.transpose()

In [ ]:
steam_dataT.head()

,name,release_date,required_age,price,dlc_count,detailed_description,about_the_game,short_description,reviews,header_image,...,positive,negative,estimated_owners,average_playtime_forever,average_playtime_2weeks,median_playtime_forever,median_playtime_2weeks,discount,peak_ccu,tags
2539430,Black Dragon Mage Playtest,"Aug 1, 2023",0,0.0,0,,,,,https://shared.akamai.steamstatic.com/store_it...,...,0,0,0 - 0,0,0,0,0,0,0,[]
496350,Supipara - Chapter 1 Spring Has Come!,"Jul 29, 2016",0,5.24,0,"Springtime, April: when the cherry trees come ...","Springtime, April: when the cherry trees come ...","Spring has come, and our protagonist, Yukinari...",,https://shared.akamai.steamstatic.com/store_it...,...,252,3,0 - 20000,8,0,8,0,65,0,"{'Adventure': 27, 'Visual Novel': 19, 'Anime':..."
1034400,Mystery Solitaire The Black Raven,"May 6, 2019",0,4.99,0,"Immerse yourself in the most beloved, mystical...","Immerse yourself in the most beloved, mystical...",Discover an entrancing and spectacular world!,,https://shared.akamai.steamstatic.com/store_it...,...,21,3,0 - 20000,0,0,0,0,0,0,"{'Casual': 83, 'Card Game': 52, 'Solitaire': 4..."
3292190,버튜버 파라노이아 - Vtuber Paranoia,"Oct 31, 2024",0,8.99,1,"synopsis 'Hello, I'm Hiyoro, a new YouTuber!' ...","synopsis 'Hello, I'm Hiyoro, a new YouTuber!' ...",Yuha! I'll start the broadcast! Hakko's extrem...,,https://shared.akamai.steamstatic.com/store_it...,...,0,0,0 - 20000,0,0,0,0,0,1,[]
3631080,Maze Quest VR,"Apr 24, 2025",0,4.99,0,Its not just a Maze; its a Quest! Enter the ca...,Its not just a Maze; its a Quest! Enter the ca...,Its not just a Maze; its a Quest! Enter the ca...,,https://shared.akamai.steamstatic.com/store_it...,...,0,0,0 - 20000,0,0,0,0,0,0,[]


In [ ]:
steam_dataT.info()

<class 'pandas.core.frame.DataFrame'>
Index: 142431 entries, 2539430 to 4516510
Data columns (total 42 columns):
 #   Column                    Non-Null Count   Dtype 
---  ------                    --------------   ----- 
 0   name                      142431 non-null  object
 1   release_date              142431 non-null  object
 2   required_age              142431 non-null  object
 3   price                     142431 non-null  object
 4   dlc_count                 142431 non-null  object
 5   detailed_description      142431 non-null  object
 6   about_the_game            142431 non-null  object
 7   short_description         142431 non-null  object
 8   reviews                   142431 non-null  object
 9   header_image              142431 non-null  object
 10  website                   142431 non-null  object
 11  support_url               142431 non-null  object
 12  support_email             142431 non-null  object
 13  windows                   142431 non-null  object
 14  ma

# Preparación de los datos

In [ ]:
# Eliminamos columnas que no tienen información de interés

to_drop = ['header_image', 'website', 'support_url', 'support_email', 'metacritic_url', 'windows', 'mac', 'linux', 'screenshots', 'movies', 'score_rank']

steam_data_prepared = steam_dataT.drop(to_drop, axis=1)

In [ ]:
steam_data_prepared.info()

<class 'pandas.core.frame.DataFrame'>
Index: 142431 entries, 2539430 to 4516510
Data columns (total 31 columns):
 #   Column                    Non-Null Count   Dtype 
---  ------                    --------------   ----- 
 0   name                      142431 non-null  object
 1   release_date              142431 non-null  object
 2   required_age              142431 non-null  object
 3   price                     142431 non-null  object
 4   dlc_count                 142431 non-null  object
 5   detailed_description      142431 non-null  object
 6   about_the_game            142431 non-null  object
 7   short_description         142431 non-null  object
 8   reviews                   142431 non-null  object
 9   metacritic_score          142431 non-null  object
 10  achievements              142431 non-null  object
 11  recommendations           142431 non-null  object
 12  notes                     142431 non-null  object
 13  supported_languages       142431 non-null  object
 14  fu

In [ ]:
# Hacemos cambios de tipo de datos

steam_data_prepared['name'] = steam_data_prepared['name'].astype('string')
steam_data_prepared['release_date'] = pd.to_datetime(steam_data_prepared['release_date'])
steam_data_prepared['required_age'] = steam_data_prepared['required_age'].astype('int32')
steam_data_prepared['price'] = steam_data_prepared['price'].astype('float32')
steam_data_prepared['dlc_count'] = steam_data_prepared['dlc_count'].astype('int32')
steam_data_prepared['detailed_description'] = steam_data_prepared['detailed_description'].astype('string')
steam_data_prepared['about_the_game'] = steam_data_prepared['about_the_game'].astype('string')
steam_data_prepared['short_description'] = steam_data_prepared['short_description'].astype('string')
steam_data_prepared['reviews'] = steam_data_prepared['reviews'].astype('string')
steam_data_prepared['metacritic_score'] = steam_data_prepared['metacritic_score'].astype('int32')
steam_data_prepared['achievements'] = steam_data_prepared['achievements'].astype('int32')
steam_data_prepared['recommendations'] = steam_data_prepared['recommendations'].astype('int32')
steam_data_prepared['notes'] = steam_data_prepared['notes'].astype('string')
#Dejamos supported_languages, full_audio_languages y packages como "object"
steam_data_prepared['developers'] = steam_data_prepared['developers'].astype('string')
steam_data_prepared['publishers'] = steam_data_prepared['publishers'].astype('string')
steam_data_prepared['categories'] = steam_data_prepared['categories'].astype('string')
steam_data_prepared['genres'] = steam_data_prepared['genres'].astype('string')
steam_data_prepared['user_score'] = steam_data_prepared['user_score'].astype('int32')
steam_data_prepared['positive'] = steam_data_prepared['positive'].astype('int32')
steam_data_prepared['negative'] = steam_data_prepared['negative'].astype('int32')

steam_data_prepared['average_playtime_forever'] = steam_data_prepared['average_playtime_forever'].astype('int32')
steam_data_prepared['average_playtime_2weeks'] = steam_data_prepared['average_playtime_2weeks'].astype('int32')
steam_data_prepared['median_playtime_forever'] = steam_data_prepared['median_playtime_forever'].astype('int32')
steam_data_prepared['median_playtime_2weeks'] = steam_data_prepared['median_playtime_2weeks'].astype('int32')

steam_data_prepared['discount'] = steam_data_prepared['discount'].astype('int32')
steam_data_prepared['peak_ccu'] = steam_data_prepared['peak_ccu'].astype('int32')

# Dejamos tags como objecto tambien

In [ ]:
steam_data_prepared.info()

In [ ]:
## Cálculo de estadísticas para deducir la unidad de MEDIDA DE TIEMPO

# Seleccionamos las columnas de tiempo de juego
playtime_cols = ['average_playtime_forever', 'average_playtime_2weeks', 'median_playtime_forever', 'median_playtime_2weeks']

# Estadísticas descriptivas principales
print("--- Estadísticas descriptivas de las columnas de tiempo de juego ---")
display(steam_data_prepared[playtime_cols].describe())

# Valores máximos de cada columna
print("\n--- Verificación de máximos absolutos ---")
for col in playtime_cols:
    max_val = steam_data_prepared[col].max()
    max_hours = max_val / 60
    print(f"{col:25} -> Máximo: {max_val:9,} minutos ({max_hours:7,.2f} horas)")

--- Estadísticas descriptivas de las columnas de tiempo de juego ---


,average_playtime_forever,average_playtime_2weeks,median_playtime_forever,median_playtime_2weeks
count,1.424310e+05,142431.000000,1.424310e+05,142431.000000
mean,1.793571e+02,12.097114,1.496984e+02,12.927776
std,1.040821e+04,251.112541,1.039408e+04,273.596924
min,0.000000e+00,0.000000,0.000000e+00,0.000000
25%,0.000000e+00,0.000000,0.000000e+00,0.000000
50%,0.000000e+00,0.000000,0.000000e+00,0.000000
75%,0.000000e+00,0.000000,0.000000e+00,0.000000
max,3.429544e+06,20088.000000,3.429544e+06,20088.000000



--- Verificación de máximos absolutos ---
average_playtime_forever  -> Máximo: 3,429,544 minutos (57,159.07 horas)
average_playtime_2weeks   -> Máximo:    20,088 minutos ( 334.80 horas)
median_playtime_forever   -> Máximo: 3,429,544 minutos (57,159.07 horas)
median_playtime_2weeks    -> Máximo:    20,088 minutos ( 334.80 horas)


# Análisis descriptivo de los datos

## Variables String y Object (con PANDAS)



In [ ]:
# Seleccionamos las columnas cuyo tipo sea 'string' u 'object'
string_obj_cols = steam_data_prepared.select_dtypes(include=['string', 'object']).columns.tolist()

print("--- Variables de tipo String y Object ---")
print(string_obj_cols)

--- Variables de tipo String y Object ---
['name', 'detailed_description', 'about_the_game', 'short_description', 'reviews', 'notes', 'supported_languages', 'full_audio_languages', 'packages', 'developers', 'publishers', 'categories', 'genres', 'estimated_owners', 'tags', 'name_normalized']


In [ ]:
# 2. Resumen estadístico básico
print("\n--- Resumen General ---")
display(steam_data_prepared[string_obj_cols].describe(include='all'))


--- Resumen General ---


In [ ]:
# 3. Análisis de valores nulos o vacíos
print("\n--- Análisis de valores vacíos y nulos ---")
vacios_df = pd.DataFrame({
    'Valores Nulos (NaN)': steam_data_prepared[string_obj_cols].isna().sum(),
    'Cadenas Vacías ("")': (steam_data_prepared[string_obj_cols] == "").sum(),
    'Colecciones Vacías (listas/diccionarios)': steam_data_prepared[string_obj_cols].apply(
        lambda col: col.apply(lambda x: len(x) == 0 if isinstance(x, (list, dict)) else False)
    ).sum()
})
display(vacios_df)


--- Análisis de valores vacíos y nulos ---


,Valores Nulos (NaN),"Cadenas Vacías ("""")",Colecciones Vacías (listas/diccionarios)
name,0,1,0
detailed_description,0,8485,0
about_the_game,0,8516,0
short_description,0,8332,0
reviews,0,129818,0
notes,0,115330,0
supported_languages,0,0,8400
full_audio_languages,0,0,84934
packages,0,0,28218
developers,0,0,0


In [ ]:
# 4. Longitud promedio de los textos descriptivos principales
print("\n--- Estadísticas de longitud de caracteres de los textos ---")
text_cols = ['name', 'detailed_description', 'about_the_game', 'short_description', 'reviews']
for col in text_cols:
    if col in steam_data_prepared.columns:
        lengths = steam_data_prepared[col].fillna("").str.len()
        print(f"{col:25} -> Media: {lengths.mean():7.1f} | Mediana: {lengths.median():7.1f} | Máx: {lengths.max():9,d}")


--- Estadísticas de longitud de caracteres de los textos ---
name                      -> Media:    17.9 | Mediana:    15.0 | Máx:       413
detailed_description      -> Media:  1279.1 | Mediana:  1052.0 | Máx:   122,702
about_the_game            -> Media:  1258.6 | Mediana:  1042.0 | Máx:   122,702
short_description         -> Media:   188.6 | Mediana:   205.0 | Máx:       342
reviews                   -> Media:    30.0 | Mediana:     0.0 | Máx:     2,913


In [ ]:
# 5. Top 10 valores más frecuentes para columnas categóricas
print("\n--- Top 10 valores más comunes por categoría ---")
categorical_text_cols = ['developers', 'publishers', 'genres', 'categories']
for col in categorical_text_cols:
    if col in steam_data_prepared.columns:
        print(f"\n[Top 10 en {col.upper()}]")
        print(steam_data_prepared[col].value_counts().head(10))


--- Top 10 valores más comunes por categoría ---

[Top 10 en DEVELOPERS]
developers
[]                              8453
['EroticGamesClub']              240
['Choice of Games']              184
['Bully Revenge Studios']        164
['Boogygames Studios']           158
['Laush Dmitriy Sergeevich']     151
['Cyber Keks']                   149
['Gamesforgames']                144
['Creobit']                      137
['King Kong']                    130
Name: count, dtype: Int64

[Top 10 en PUBLISHERS]
publishers
[]                       8852
['BFG Entertainment']     554
['8floor']                268
['EroticGamesClub']       240
['Strategy First']        213
['Kagura Games']          213
['Conglomerate 5']        197
['HH-Games']              196
['Hede']                  188
['Choice of Games']       184
Name: count, dtype: Int64

[Top 10 en GENRES]
genres
[]                                  8435
['Casual', 'Indie']                 8158
['Action', 'Indie']                 6385
['Action

## Variables int y float (con POLARS)

In [ ]:
import polars as pl

# Definimos las columnas numéricas de interés
numeric_cols = ['required_age', 'price', 'dlc_count', 'metacritic_score', 'achievements',
                'recommendations', 'user_score', 'positive', 'negative',
                'average_playtime_forever', 'average_playtime_2weeks',
                'median_playtime_forever', 'median_playtime_2weeks', 'discount', 'peak_ccu']

# Creamos el DataFrame de Polars
pl_df = pl.from_pandas(steam_data_prepared[numeric_cols])

# Calculamos el resumen descriptivo en Polars
summary_pl = pl_df.describe()

# Redondeo a 3 decimales
summary_pl = summary_pl.with_columns([
    pl.col(c).cast(pl.Float64).round(3) if pl_df[c].dtype.is_numeric() else pl.col(c)
    for c in numeric_cols
])

display(summary_pl.to_pandas())

,statistic,required_age,price,dlc_count,metacritic_score,achievements,recommendations,user_score,positive,negative,average_playtime_forever,average_playtime_2weeks,median_playtime_forever,median_playtime_2weeks,discount,peak_ccu
0,count,142431.000,142431.000,142431.000,142431.000,142431.000,142431.000,142431.000,142431.000,142431.00,142431.000,142431.000,142431.000,142431.000,142431.000,142431.000
1,null_count,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.00,0.000,0.000,0.000,0.000,0.000,0.000
2,mean,0.151,4.993,0.501,2.219,17.484,833.625,0.021,899.764,145.72,179.357,12.097,149.698,12.928,15.945,46.998
3,std,1.571,12.425,13.529,12.737,131.508,20303.615,1.294,26066.497,4987.04,10408.207,251.113,10394.080,273.597,27.497,3460.299
4,min,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.00,0.000,0.000,0.000,0.000,0.000,0.000
5,25%,0.000,0.690,0.000,0.000,0.000,0.000,0.000,0.000,0.00,0.000,0.000,0.000,0.000,0.000,0.000
6,50%,0.000,2.690,0.000,0.000,3.000,0.000,0.000,2.000,0.00,0.000,0.000,0.000,0.000,0.000,0.000
7,75%,0.000,5.990,0.000,0.000,19.000,0.000,0.000,26.000,7.00,0.000,0.000,0.000,0.000,30.000,0.000
8,max,21.000,999.990,3703.000,97.000,9821.000,4830455.000,100.000,7642084.000,1173003.00,3429544.000,20088.000,3429544.000,20088.000,100.000,1013936.000


In [ ]:
# 2. Métricas adicionales de calidad de datos calculadas puramente en POLARS
total_rows = pl_df.height

# Calculamos nulos, ceros y sus porcentajes utilizando expresiones nativas de Polars
quality_exprs = []
for col in numeric_cols:
    quality_exprs.extend([
        pl.col(col).null_count().alias(f"{col}_nulos"),
        (pl.col(col).null_count() / total_rows * 100).round(2).alias(f"{col}_pct_nulos"),
        (pl.col(col) == 0).sum().alias(f"{col}_ceros"),
        ((pl.col(col) == 0).sum() / total_rows * 100).round(2).alias(f"{col}_pct_ceros")
    ])

# Ejecutamos las agregaciones en un solo paso eficiente
quality_stats = pl_df.select(quality_exprs)

# Reestructuramos el resultado para tenerlo como tabla legible por columna
quality_rows = []
for col in numeric_cols:
    row = {
        "Variable": col,
        "Valores Nulos (NaN)": quality_stats.get_column(f"{col}_nulos")[0],
        "% Nulos": quality_stats.get_column(f"{col}_pct_nulos")[0],
        "Valores Cero (0)": quality_stats.get_column(f"{col}_ceros")[0],
        "% Ceros": quality_stats.get_column(f"{col}_pct_ceros")[0]
    }
    quality_rows.append(row)

quality_metrics_pl = pl.DataFrame(quality_rows)
print("--- Métricas de Calidad de Datos con Polars ---")
display(quality_metrics_pl.to_pandas())

--- Métricas de Calidad de Datos con Polars ---


,Variable,Valores Nulos (NaN),% Nulos,Valores Cero (0),% Ceros
0,required_age,0,0.0,141099,99.06
1,price,0,0.0,28615,20.09
2,dlc_count,0,0.0,122101,85.73
3,metacritic_score,0,0.0,138155,97.00
4,achievements,0,0.0,67984,47.73
5,recommendations,0,0.0,120656,84.71
6,user_score,0,0.0,142391,99.97
7,positive,0,0.0,62060,43.57
8,negative,0,0.0,74983,52.65
9,average_playtime_forever,0,0.0,116160,81.56


In [ ]:
# 3. Tabla descriptiva completa unificada
# Transponemos la tabla
summary_long = summary_pl.unpivot(index="statistic", on=numeric_cols)
summary_transposed = summary_long.pivot(on="statistic", index="variable", values="value")
summary_transposed = summary_transposed.rename({"variable": "Variable"})

# Unimos el análisis descriptivo con las métricas de calidad de datos
complete_numeric_analysis_pl = summary_transposed.join(quality_metrics_pl, on="Variable")

print("--- Análisis Descriptivo Completo de Variables Numéricas ---")
display(complete_numeric_analysis_pl.to_pandas())

--- Análisis Descriptivo Completo de Variables Numéricas ---


,Variable,count,null_count,mean,std,min,25%,50%,75%,max,Valores Nulos (NaN),% Nulos,Valores Cero (0),% Ceros
0,required_age,142431.0,0.0,0.151,1.571,0.0,0.00,0.00,0.00,21.00,0,0.0,141099,99.06
1,price,142431.0,0.0,4.993,12.425,0.0,0.69,2.69,5.99,999.99,0,0.0,28615,20.09
2,dlc_count,142431.0,0.0,0.501,13.529,0.0,0.00,0.00,0.00,3703.00,0,0.0,122101,85.73
3,metacritic_score,142431.0,0.0,2.219,12.737,0.0,0.00,0.00,0.00,97.00,0,0.0,138155,97.00
4,achievements,142431.0,0.0,17.484,131.508,0.0,0.00,3.00,19.00,9821.00,0,0.0,67984,47.73
5,recommendations,142431.0,0.0,833.625,20303.615,0.0,0.00,0.00,0.00,4830455.00,0,0.0,120656,84.71
6,user_score,142431.0,0.0,0.021,1.294,0.0,0.00,0.00,0.00,100.00,0,0.0,142391,99.97
7,positive,142431.0,0.0,899.764,26066.497,0.0,0.00,2.00,26.00,7642084.00,0,0.0,62060,43.57
8,negative,142431.0,0.0,145.720,4987.040,0.0,0.00,0.00,7.00,1173003.00,0,0.0,74983,52.65
9,average_playtime_forever,142431.0,0.0,179.357,10408.207,0.0,0.00,0.00,0.00,3429544.00,0,0.0,116160,81.56


#Limpieza de datos

### Demos y Playtests

En Steam se publican, a parte de los juegos acabados, distintas versiones de juegos inacabados, con acceso anticipado y demos. Todas estas versiones tienen unas características diferentes a los lanzamientos de juegos tradicionales y sesgarían nuestro análisis.

In [ ]:
# Filtramos y contamos juegos con "Playtest" o "Demo" en el nombre

# Creamos mascaras para filtrar palabras clave
playtest_mask = steam_data_prepared['name'].str.contains('playtest', case=False, na=False)
demo_mask = steam_data_prepared['name'].str.contains('demo', case=False, na=False)
combined_mask = playtest_mask | demo_mask

# Calculamos los totales
playtest_count = playtest_mask.sum()
demo_count = demo_mask.sum()
total_count = combined_mask.sum()

print(f"Juegos que contienen la palabra 'Playtest': {playtest_count}")
print(f"Juegos que contienen la palabra 'Demo': {demo_count}")
print(f"Juegos únicos que contienen 'Playtest' o 'Demo': {total_count}")

Juegos que contienen la palabra 'Playtest': 8144
Juegos que contienen la palabra 'Demo': 557
Juegos únicos que contienen 'Playtest' o 'Demo': 8673


In [ ]:
# Creamos el nuevo dataframe filtrando los elementos usando la negación de la máscara combinada anterior
steam_data_noplaytestdemo = steam_data_prepared[~combined_mask].copy()

# Mostramos información sobre el cambio de tamaño para verificar que se han eliminado correctamente
entries_removed = len(steam_data_prepared) - len(steam_data_noplaytestdemo)
print(f"Entradas eliminadas con éxito: {entries_removed}")
print(f"Tamaño del nuevo DataFrame (sin Playtests ni Demos): {steam_data_noplaytestdemo.shape[0]} entradas")


Entradas eliminadas con éxito: 8673
Tamaño del nuevo DataFrame (sin Playtests ni Demos): 133758 entradas


Vamos a comprobar si podemos agrupar algunas entradas duplicadas si tienen el mismo nombre y el mismo desarrollador. Una vez encontrados, nos fijamos tambien en si comparten fecha de lanzamiento u otras variables.

In [ ]:
# Comprobando si hay IDs duplicados
print("Indices duplicados:", steam_data_noplaytestdemo.index.duplicated().sum())

Indices duplicados: 0


In [ ]:
# Comprobando si hay nombres duplicados que además coincidan en "developers"
# Normalizamos la cadena de texto del nombre
steam_data_noplaytestdemo["name_normalized"] = (steam_data_noplaytestdemo["name"].str.strip().str.casefold())

# Buscamos duplicados que coincidan en las columnas "name_normalized" y "developers"
dupe_nombre_dev = steam_data_noplaytestdemo.duplicated(subset=["name_normalized", "developers"], keep=False)

print("Número de registros duplicados (mismo nombre y mismo desarrollador):", dupe_nombre_dev.sum())


Número de registros duplicados (mismo nombre y mismo desarrollador): 205


In [ ]:
# Filtramos el dataframe para mostrar los registros duplicados exactos en nombre y developer
show_dupes = steam_data_noplaytestdemo[dupe_nombre_dev].sort_values(by=["name_normalized", "developers"])

# Mostramos columnas relevantes para los registros duplicados
show_dupes[["name", "release_date", "price", "discount", "estimated_owners", "developers"]]

,name,release_date,price,discount,estimated_owners,developers
34010,Alpha Protocol™,2010-06-01,9.99,50,"(500000, 1000000)",['Obsidian Entertainment']
34019,Alpha Protocol™,2010-06-01,9.99,50,"(0, 20000)",['Obsidian Entertainment']
33230,Assassin's Creed 2,2010-03-09,4.99,75,"(2000000, 5000000)",['Ubisoft Montreal']
48173,Assassin's Creed 2,2010-03-09,4.99,75,"(0, 20000)",['Ubisoft Montreal']
35140,Batman: Arkham Asylum Game of the Year Edition,2010-03-26,3.99,0,"(2000000, 5000000)",['Rocksteady Studios']
...,...,...,...,...,...,...
240760,Wasteland 2: Director's Cut,2014-09-18,8.99,70,"(500000, 1000000)",['inXile Entertainment']
223870,Ys I & II Chronicles+,2013-02-14,4.49,70,"(100000, 200000)",['Nihon Falcom']
223810,Ys I & II Chronicles+,2013-02-14,4.49,70,"(200000, 500000)",['Nihon Falcom']
3170460,小先生 Little ShiFu,2026-09-11,5.61,0,"(0, 0)",['Liminal Factory']


In [ ]:
#TODO Hacer funcion para extraer el precio inicial de "packages"

### Variable Objetivo

Steam no proporciona datos precisos de ventas así que necesitaremos definir una variable objectivo sustitutiva que nos sirva de proxy para aproximar el éxito comercial de los lanzamientos de videojuegos.

Como disponemos de un rango de propietarios de los juegos vamos a usar la media aritmética para determinar el punto medio y usar la variable resultante cómo objectivo.

In [ ]:
#Formateo de la columna "estimated_owners"
# Aplicamos la función para pasar de strings a tuplas con el rango
steam_data_noplaytestdemo['estimated_owners'] = steam_data_noplaytestdemo['estimated_owners'].apply(parse_estimated_owners)

# Aplicamos función lambda con la media aritmética
steam_data_noplaytestdemo['estimated_owners_mid'] = steam_data_noplaytestdemo['estimated_owners'].apply(lambda x: round((x[0] + x[1]) / 2))

# Comprobamos los resultados
display(steam_data_noplaytestdemo[['name', 'estimated_owners', 'estimated_owners_mid']].head())

,name,estimated_owners,estimated_owners_mid
496350,Supipara - Chapter 1 Spring Has Come!,"(0, 20000)",10000
1034400,Mystery Solitaire The Black Raven,"(0, 20000)",10000
3292190,버튜버 파라노이아 - Vtuber Paranoia,"(0, 20000)",10000
3631080,Maze Quest VR,"(0, 20000)",10000
1654170,Agony VR,"(0, 20000)",10000


prueba